In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

In [ ]:
# ── 1. Generate the Moon dataset ──────────────────────────────────────────
np.random.seed(42)
X, y = make_moons(n_samples=500, noise=0.25, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Training samples : {X_train.shape[0]}")
print(f"Test samples     : {X_test.shape[0]}")
print(f"Features         : {X_train.shape[1]}")
print(f"Classes          : {np.unique(y)}")

In [ ]:
# ── 2. Plot the raw moon data ─────────────────────────────────────────────
plt.figure(figsize=(8, 5))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="RdBu",
            edgecolors="black", s=40, label="Train")
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap="RdBu",
            edgecolors="black", s=40, marker="^", label="Test")
plt.title("Moon Dataset — Non-linearly Separable", fontsize=14)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.tight_layout()
plt.show()

## Linear Classification (Logistic Regression)

In [ ]:
# ── 3. Logistic Regression (linear decision boundary) ─────────────────────
lin_clf = LogisticRegression(random_state=42)
lin_clf.fit(X_train, y_train)

y_pred_lin = lin_clf.predict(X_test)
acc_lin = accuracy_score(y_test, y_pred_lin)

print(f"Linear Classifier Accuracy: {acc_lin:.4f}")
print(classification_report(y_test, y_pred_lin, digits=4))

In [ ]:
# Helper: plot decision boundary
def plot_decision_boundary(clf, X, y, title, ax=None):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    grid_c = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))
    ax.contourf(xx, yy, grid_c, alpha=0.3, cmap="RdBu")
    ax.contour(xx, yy, grid_c, colors="black", linewidths=0.5)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="RdBu", edgecolors="black", s=30)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")

t = "Linear Decision Boundary  (Acc={:.4f})".format(acc_lin)
plot_decision_boundary(lin_clf, X_test, y_test, t)
plt.tight_layout()
plt.show()

## Polynomial Classification (degree 3)

In [ ]:
# ── 4. Polynomial Features + Logistic Regression ──────────────────────────
degree = 3

poly_feats = PolynomialFeatures(degree=degree, include_bias=False)
X_train_poly = poly_feats.fit_transform(X_train)
X_test_poly  = poly_feats.transform(X_test)

print(f"Original features : {X_train.shape[1]}")
print(f"Polynomial features: {X_train_poly.shape[1]}")
print(f"Feature names     : {poly_feats.get_feature_names_out()}")

In [ ]:
poly_clf = LogisticRegression(random_state=42, max_iter=10000)
poly_clf.fit(X_train_poly, y_train)

y_pred_poly = poly_clf.predict(X_test_poly)
acc_poly = accuracy_score(y_test, y_pred_poly)

print(f"Polynomial Classifier Accuracy: {acc_poly:.4f}")
print(classification_report(y_test, y_pred_poly, digits=4))

In [ ]:
t = "Polynomial Decision Boundary  (deg={}, Acc={:.4f})".format(degree, acc_poly)
plot_decision_boundary(poly_clf, X_test_poly[:, :2], y_test, t)
plt.tight_layout()
plt.show()

## Side-by-Side Comparison

In [ ]:
# ── 5. Compare both decision boundaries ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

t1 = "Linear  (Acc={:.4f})".format(acc_lin)
t2 = "Polynomial deg={}  (Acc={:.4f})".format(degree, acc_poly)

plot_decision_boundary(lin_clf, X_test, y_test, t1, ax=axes[0])
plot_decision_boundary(poly_clf, X_test_poly[:, :2], y_test, t2, ax=axes[1])

plt.suptitle("Linear vs Polynomial Classification on Moon Dataset",
             fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## Accuracy at Different Polynomial Degrees

In [ ]:
# ── 6. Sweep polynomial degrees ──────────────────────────────────────────
degrees = range(1, 9)
train_accs, test_accs = [], []

for d in degrees:
    pf = PolynomialFeatures(degree=d, include_bias=False)
    X_tr_p = pf.fit_transform(X_train)
    X_te_p = pf.transform(X_test)

    clf = LogisticRegression(random_state=42, max_iter=10000)
    clf.fit(X_tr_p, y_train)

    train_accs.append(accuracy_score(y_train, clf.predict(X_tr_p)))
    test_accs.append(accuracy_score(y_test, clf.predict(X_te_p)))

plt.figure(figsize=(9, 5))
plt.plot(degrees, train_accs, "bo-", label="Train Accuracy", linewidth=2)
plt.plot(degrees, test_accs, "rs-", label="Test Accuracy", linewidth=2)
plt.xlabel("Polynomial Degree", fontsize=12)
plt.ylabel("Accuracy", fontsize=12)
plt.title("Effect of Polynomial Degree on Moon Dataset", fontsize=14)
plt.xticks(degrees)
plt.ylim(0.5, 1.05)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print()
print("Degree | Train Acc | Test Acc | # Features")
print("-" * 50)
for d in degrees:
    pf = PolynomialFeatures(degree=d, include_bias=False)
    n_feat = pf.fit_transform(X_train[:1]).shape[1]
    print("  {}    |   {:.4f}  |  {:.4f}  |    {}".format(
        d, train_accs[d-1], test_accs[d-1], n_feat))